# {{TITLE}}

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fbcouto/multimerge-genomics-research/blob/main/notebooks/{{NOTEBOOK_PATH}})

**The problem.** *One paragraph: the real-world situation, the question, and why it is not trivial.*

**What you need.** *The input files and what they must contain.*

**What you get.** *The output files, tables and plots.*

In [ ]:
# Setup. Runs on Google Colab or on your own computer: on Colab, the first run installs MMGEN from the GitHub
# release (about a minute). Tables use pandas and plots matplotlib when installed (Colab has both); without
# them the notebook prints plain text and skips the plots.
import platform, subprocess, sys
from pathlib import Path

RELEASE = "https://github.com/fbcouto/multimerge-genomics-research/releases/download/v2.2.0"
try:
    import multimerge_genomics as mmg
except ImportError:
    if platform.system() != "Linux":
        print("MMGEN is not installed in this Python. Download the wheel for your system from\n"
              "  https://github.com/fbcouto/multimerge-genomics-research/releases/tag/v2.2.0\n"
              "install it with:  python -m pip install <the .whl file>\n"
              "and run this cell again.")
        raise
    print("Installing MMGEN (about a minute)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{RELEASE}/multimerge_genomics-2.2.0-cp310-abi3-manylinux_2_28_x86_64.whl"], check=True)
    import multimerge_genomics as mmg

IN_NOTEBOOK = "ipykernel" in sys.modules
try:
    from IPython.display import display, Markdown
except ImportError:                        # plain Python: print instead
    display, Markdown = print, str
try:
    import pandas as pd
    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", 30)
except ImportError:
    pd = None
try:
    import matplotlib
    if not IN_NOTEBOOK:
        matplotlib.use("Agg")              # plain Python: save the figures, open no window
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

# Chart colors (a palette checked for color-blind readers): two series, a two-sided scale, neutral ink
BLUE, ORANGE, RED, GRAY, INK = "#2a78d6", "#eb6834", "#e34948", "#a3a29c", "#52514e"


def show(rows, n=25):
    """Shows a list of dictionaries as a table (pandas when available, plain text otherwise)."""
    rows = list(rows)
    if pd is not None:
        display(pd.DataFrame(rows[:n]))
    else:
        for row in rows[:n]:
            print("  ".join(f"{k}: {v}" for k, v in row.items()))
    if len(rows) > n:
        print(f"... and {len(rows) - n} more rows")


def tidy(ax):
    """Recessive axes: no top or right frame, a light grid behind the data."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)


def save_figure(fig, path):
    """Saves a figure; shows it in a notebook, closes it in plain Python."""
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("Figure saved to", path)
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close(fig)


print(mmg.engine_info())

## Data

*Which demo data this notebook uses by default, where it comes from (source, license; for human data, that
the people consented to open sharing), and which variable to set to use your own files.*

In [ ]:
# Your files (None = the demo data)
MY_INPUT = None              # e.g. "my_file.vcf.gz"

import urllib.request, zipfile

DATA = Path("data")                          # the demo files are extracted here
OUT = Path("results") / "{{NAME}}"         # everything this notebook writes
OUT.mkdir(parents=True, exist_ok=True)


def demo_file(pack, member):
    """One file of a demo pack: downloads the pack once into the working folder and extracts only that file."""
    target = DATA / member
    if not target.exists():
        if not Path(pack).exists():
            print(f"Downloading {pack} from the MMGEN release...")
            urllib.request.urlretrieve(f"{RELEASE}/{pack}", pack + ".part")
            Path(pack + ".part").replace(pack)
        with zipfile.ZipFile(pack) as z:
            z.extract(member, DATA)
    return str(target)


USING_DEMO = MY_INPUT is None
# The demo file: a pack ("mmgen_demo_human_v3.zip" or "mmgen_demo_research_v4.zip") and a file inside it (see the pack's README.md)
INPUT = MY_INPUT or demo_file("mmgen_demo_research_v4.zip", "research/outbreak/NC_045512.2.fna.gz")
print("Input:", INPUT)

## Step 1. *What this step does*

*Why: the method in one short paragraph, and what can go wrong.*

In [ ]:
# Call MMGEN and print the key numbers. Every function, with its parameters:
#   from multimerge_genomics import ai; [print(t["signature"]) for t in ai.catalog()]   and   help(mmg.<function>)
report = mmg.validate_file(INPUT)
print(report["text"])

## Step 2. *What this step does*

*Why.*

In [ ]:
# The next call, e.g.  mmg.sort_genomic([INPUT], str(OUT / "sorted.vcf.gz"), index=True)

## Check the result

*What a reviewer would ask: counts that must agree, files that must exist, values in a plausible range.*

In [ ]:
def check(condition, text):
    print(("OK     " if condition else "CHECK  ") + text)


check(report["valid"], "the input is valid")
if USING_DEMO:
    pass   # assert the numbers the demo data must give

## Methods paragraph

A starting text for the methods section of a paper or report, filled in with the version and the parameters
this run used. Edit it to your study, and cite the methods MMGEN reproduces (see References).

In [ ]:
import textwrap
VERSION = mmg.engine_info()["version"]
methods = f"""*What was done, with the functions and the parameters used.* All steps used MMGEN version {VERSION}
(Couto F. MMGEN: MultiMerge Genomics (version {VERSION}). Zenodo. https://doi.org/10.5281/zenodo.20417492)."""
print(textwrap.fill(" ".join(methods.split()), 110))

## Adapt it to your data

* *The parameters to change, and when.*

## References

* *The methods and tools each step reproduces, and the source of the demo data.*